In [24]:
# Libraries
from pypdf import PdfReader
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown
import gradio as gr



In [37]:
# Load variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [38]:
# Call API
def call_gpt(pdf, system_prompt, user_prompt):
    # Create model instance
    openai = OpenAI()
    response = openai.chat.completions.create(
        model='gpt-5-mini',
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content" : user_prompt + pdf}
        ],
        reasoning_effort='medium'
    )
    return response.choices[0].message.content


In [39]:
# Read PDF
def read_pdf(pdf_file):

    reader = PdfReader(pdf_file.name)
    text = []

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text.append(content)

    return "\n\n".join(text)

In [50]:
# Create promts
def main(pdf_file):
    system_prompt = """ Rol sistema – Agente revisor de especificaciones funcionales SAP ABAP / S/4HANA
Sos un arquitecto SAP senior especializado en S/4HANA, ABAP, Clean Core, Fiori, RAP, CDS, integración y buenas prácticas de diseño.
Tu tarea es analizar una especificación funcional recibida por un equipo ABAP y actuar como revisor técnico-funcional crítico.
Debés:
Analizar la coherencia general de la especificación.
Detectar inconsistencias funcionales, técnicas y arquitectónicas.
Identificar ambigüedades, omisiones, contradicciones y definiciones incompletas.
Señalar errores en el mapeo de tablas y campos, relaciones entre documentos y joins propuestos.
Evaluar si la solución propuesta es adecuada para el contexto tecnológico del cliente.
Cuestionar explícitamente decisiones que no sean consistentes con:
SAP S/4HANA
Fiori
Clean Core
performance
escalabilidad
mantenibilidad
Proponer una arquitectura de solución superadora, aunque contradiga la especificación.
Explicar por qué la arquitectura propuesta es mejor que la originalmente pedida.
Priorizar opciones modernas como:
CDS Views
RAP
Fiori Elements
APIs released
extensibilidad side-by-side o key-user cuando aplique
Desaconsejar explícitamente enfoques legacy cuando no correspondan, por ejemplo:
ALV clásico como solución principal para usuarios Fiori
acceso directo indiscriminado a tablas
desarrollos que violen clean core
lógicas no alineadas con el modelo de S/4HANA
Instrucciones específicas de análisis
Cuando recibas la especificación, devolvé tu respuesta en las siguientes secciones:
A. Resumen ejecutivo
Qué intenta resolver la EF
Si la solución propuesta es viable o no
Riesgo general: bajo / medio / alto
B. Hallazgos críticos
Listá todos los errores o problemas detectados, separados por categoría:
Arquitectura
Funcional
Datos y modelo
Tablas y campos
Performance
Seguridad/autorizaciones
UX / Fiori
Ambigüedades y faltantes
C. Errores de mapeo
Armá una tabla con:
Campo requerido
Tabla/campo indicado en la EF
Problema detectado
Tabla/campo o enfoque sugerido
Justificación
D. Contradicciones
Indicá dónde la EF se contradice a sí misma.
E. Preguntas que el equipo debería devolver al funcional
Generá una lista priorizada de preguntas para cerrar gaps.
F. Arquitectura propuesta
Definí la arquitectura recomendada indicando, según corresponda:
si mantener o no el ALV clásico
si reemplazar por RAP/Fiori
si conviene usar CDS
si conviene separar modo resumen y detalle
cómo resolver estados
cómo resolver performance
qué objetos ABAP crearías
G. Recomendación final
Respondé con una conclusión concreta, por ejemplo:
“No recomiendo construir esta solución como ALV clásico”
“Propongo rediseñar la solución como aplicación Fiori con backend RAP”
“La EF no está lista para desarrollo y requiere aclaraciones previas”
Estilo de respuesta
Sé riguroso y crítico.
No asumas que la EF está bien.
No intentes salvar una mala especificación sin explicarlo.
Si encontrás algo incorrecto, decilo de forma directa.
Priorizá precisión técnica por sobre diplomacia.
Si hay varias alternativas, recomendá una y explicá por qué. """

    user_prompt = f""" 
    Responde exclusivamente en Markdown valido
    
    Rol desarrollador – Analista ABAP senior que recibe una EF y debe prepararla para construcción
Sos un desarrollador ABAP senior en SAP S/4HANA, con experiencia en:
SD
ALV
CDS Views
RAP
Fiori
performance
debugging funcional-técnico
análisis de especificaciones funcionales defectuosas
Tu tarea es recibir una especificación funcional y evaluarla como si fueras el responsable técnico de construir la solución.
No debés limitarte a programar lo pedido.
Debés cuestionar la especificación y transformarla en una solución técnicamente consistente.
Objetivos
Detectar errores funcionales y técnicos.
Revisar si el requerimiento es implementable tal como está.
Identificar mapeos incorrectos de tablas y campos.
Detectar joins inválidos o peligrosos.
Señalar ambigüedades que impedirían desarrollar.
Proponer una solución técnica viable.
Indicar el impacto de la solución en S/4HANA y Fiori.
Definir qué deberías aclarar antes de estimar o desarrollar.
Instrucciones de salida
Respondé con esta estructura:
1. Entendimiento del requerimiento
Explicá brevemente qué pide el usuario de negocio.
2. Problemas detectados para desarrollar
Separá:
Bloqueantes
Riesgos altos
Riesgos medios
Supuestos peligrosos
3. Revisión del modelo de datos
Para cada dato importante del reporte, indicá:
si el origen propuesto parece correcto o incorrecto
qué validaciones harías
qué relación documental revisarías
4. Revisión de arquitectura técnica
Indicá si construirías:
un reporte ALV clásico
un ALV OO
una CDS + app Fiori
una solución RAP
una solución híbrida
Justificá la elección según:
contexto S/4
usuarios Fiori
mantenibilidad
performance
clean core
5. Lista de aclaraciones necesarias
Generá preguntas concretas para el funcional, agrupadas por tema:
alcance
datos
estados
pricing
performance
UX
autorizaciones
6. Propuesta técnica
Definí una propuesta concreta incluyendo:
objetos a crear
estrategia de lectura de datos
estrategia para resolver estados
estrategia para documentos múltiples
estrategia para performance
estrategia para navegación o drill-down
7. Estimación condicionada
Indicá:
si la EF permite estimar o no
qué partes no pueden estimarse todavía
qué dependencias funcionales faltan cerrar
Criterios de evaluación
No aceptes mapeos incorrectos sin marcarlos.
No aceptes una arquitectura vieja solo porque la EF la pide.
Si el negocio pide ALV clásico pero el contexto indica Fiori, debés señalar el desvío.
Si hay mezcla entre ECC y S/4 sin validación, debés advertirlo.
Si el flujo SD/FI está mal simplificado, debés marcarlo.
Si faltan definiciones para estados o pricing, debés marcarlo como bloqueo.
Estilo
Técnico, claro y directo.
Pensá como alguien que luego tendrá que desarrollar, probar, documentar y defender la solución.
No supongas datos que la EF no define.
Indicá expresamente cuándo algo “no alcanza para construir”.

Este es el texto de la definición funcional:

 """

    pdf_txt = read_pdf(pdf_file)
    if not pdf_txt:
        return 'No se pudo extraer texto del PDF'

    return call_gpt(pdf_txt, system_prompt, user_prompt)

In [51]:
def launch_interface():
    view = gr.Interface(
        fn=main,
        inputs=gr.File(file_types=[".pdf"], label="Arrastrá el PDF del requerimiento"),
        outputs=gr.Markdown(label="Texto extraído"),
        title="Revisor de requerimientos ABAP",
        flagging_mode="never"
    )

    view.launch(inbrowser=True)

if __name__ == "__main__":
    launch_interface()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Output


1. Entendimiento del requerimiento
- El usuario pide un reporte Z para seguimiento comercial que muestre en una sola pantalla la trazabilidad desde pedido de venta → entrega → PGI → factura → cobro, con filtros y acciones (doble-clic para abrir factura, editar fecha de entrega desde la grilla) y semáforos por estado. Actualmente piden ALV clásico ejecutable desde SAP GUI, pero quieren posibilidad de usarlo desde Fiori. Volumen estimado 50-100k registros por consulta; tiempo objetivo < 2 segundos.

2. Problemas detectados para desarrollar

Bloqueantes (impiden construir)
- Mapeos de tablas y joins propuestos incorrectos/erróneos (ver sección 3 y 4). Con los joins actuales no se obtendrán los datos coherentes.
- Reglas de negocio incompletas: definición de “estado general” no está completa (prioridad entre VBUK/VBUP/estado de factura); qué es “pedido rechazado” y “documentos anulados” no está definido (campos y valores concretos).
- Requisitos funcionales contradictorios: piden ALV clásico y “mantener mismo desarrollo para SAP GUI y Fiori” sin definir interfaz Fiori ni restricción de UX/UX patterns.
- Requisito de edición directa de fecha de entrega en grilla sin especificar validaciones, autorización ni método (actualizar LIKP/LIPS o documento de entrega?) — riesgo de violar clean core y de inconsistencias con logística.

Riesgos altos
- Performance: 50-100k registros por consulta y requisito < 2s es poco realista con ALV clásico leyendo múltiples tablas SD/FI con joins complejos sin índices/aggergaciones. Riesgo alto de timeouts y carga en BD.
- Integridad de datos FI: se menciona “cobro automático” y uso de BSID/BSEG/BKPF; mapping actual usa BSID.BUDAT para fecha de cobro (BSID es partidas abiertas cliente — no siempre tiene BUDAT de compensación; compensación está en BSEG o en tabla REGUH/BSIK/BSID según documento). Riesgo de traer datos incorrectos.
- Uso de tablas cluster KONV sin método correcto: leer KONV directamente es frágil en S/4HANA (clustered, moved data) y puede generar pésima performance y discrepancias en S/4.

Riesgos medios
- Autorizaciones no definidas (objeto, roles, check lógico). Riesgo de exponer edición indebida.
- La idea de “una sola línea por pedido” vs posibilidad de ver detalle por posición no está claro cómo lo espera el usuario (agrupamiento vs drill-down). Riesgo de UX confusa.

Supuestos peligrosos (a cuestionar)
- “Modelo de datos S/4 mantiene mismas tablas y lógica que ECC” — falso en muchos casos (por ejemplo, ACDOCA, cambios en KONV, MATDOC, CDS). No se debe asumir igualdad.
- “Acceder directamente a tablas estándar para simplificar” — contradice clean core y buenas prácticas S/4; peligro de romper upgradeability y performance.

3. Revisión del modelo de datos (por dato requerido)
Para cada dato clave indico si origen propuesto es correcto, validaciones y relaciones documentales a revisar.

- Pedido de venta (VBELN)
  - Origen: VBAK (header) OK para número de pedido.
  - Validaciones: tipo de doc (VBAK-AUART), status de cabecera (VBUK), marca de borrado (VBAK-LFSTK? o flag de cancelación?).
  - Relación a revisar: VBAK -> VBAP (items) por VBELN.

- Posición de pedido (POSNR)
  - Origen propuesto: VBAK.POSNR — incorrecto. POSNR está en VBAP (tabla de posiciones). VBAK no tiene POSNR.
  - Validación: existencia en VBAP, estado por posición en VBUP.

- Cliente / Nombre cliente
  - KUNNR en VBAK/VBPA/KNA1: usar KUNNR desde VBAK (cabecera) y textos desde KNA1/KNB1/KNA1. VBPA es partner por documento y rol; NAME1 en VBPA puede no ser fiable (VBPA contiene partners por rol).
  - Validar: sales area (KNVV), partner functions (AG/WE/VB) para vendedor.

- Material / Descripción
  - Material correcto desde VBAP-MATNR (pos. pedido), no VBRP. MAKT(MAKTX) es tabla texto por idioma; usar asociación por MATNR + SPRAS.
  - Validar: unidad (VBAP-VRKME).

- Cantidad pedida
  - Origen VBAK-KWMENG es cantidad total de cabecera — es inconsistente si se necesita por posición. Si detalle por posición usar VBAP-KWMENG.
  - Validar: manejo de unidades por posición.

- Cantidad entregada
  - Origen correcto: LIPS-LFIMG (posiciones de entrega). No está en LIPS si se quiere por pedido: agregar SUM(LIPS-LFIMG) por VBELN_VBELN (referencia al pedido) o usar campo VBAP-LFIMG (cantidad ya entregada por posición).
  - Validar: consideraciones de devoluciones, transferencias entre entregas.

- Fecha de PGI
  - Origen propuesto MKPF-BUDAT — incorrecto para date de salida de entrega: PGI date está en LIPS-WADAT_IST (o LIKP-WADAT_IST en header) o LIKP-GM_BUDAT/LIKP-BUDAT (dependiendo). MKPF-BUDAT es fecha de documento material (puede coincidir) pero JOIN a MKPF sin material doc link es erróneo.

- Entrega / Posición entrega
  - LIKP (header) y LIPS (items): PROPONER LIKP-VBELN (delivery header) y LIPS-VBELN (delivery item). No use LIKP.POSNR; posición en LIKP no existe.

- Factura / Posición factura / Cantidad facturada
  - VBRK (cabecera) y VBRP (posiciones). Cantidad facturada está en VBRP-FKIMG. VBRK-FKIMG no es correcto (VBRK no tiene FKIMG por posición).
  - Validar: facturas de crédito, notas de abono, facturas canceladas.

- Importe neto / condición de precio
  - Propuesto KONV-KBETR: leer KONV directamente es desaconsejado. Mejor usar CDS que exponga pricing o usar BAPI_PRICES or view de condiciones; en S/4HANA KONV puede no reflejar documentos históricos igual.
  - Validar: qué condición se requiere (PR00? importe neto? subtotal?), moneda.

- Moneda
  - VBAK-WAERK OK.

- Estados (VBUK, VBUP, VBUK.FKSTK, VBUK.GBSTK)
  - VBUK contiene campos de status de cabecera; VBUP para items. Mapping propuesto en líneas está parcialmente correcto, pero se requiere definición de prioridad y valores específicos. Además estados de factura típicamente en VBRK-FKART/FLAGs o en VBUK-FKSTK? VBUK-FKSTK es bandera de facturación, pero cuidado con interpretación.

- Documento contable / Fecha de cobro
  - BKPF.BELNR mapping incorrecto: BKPF contiene FI document header linked por BELNR+BUKRS+GJAHR; VBRK-BELNR no es igual a BKPF-BELNR necesariamente sin clave de compañía y ejercicio. Para partidas compensadas ver tabla BSAD (partidas compensadas) o BSEG con cabecera BKPF. BSID es partidas abiertas; fecha de compensación (fecha de clearing) está en BSAD-BUDAT o BSEG-BUDAT según compensación.
  - Validar: cómo se detecta “pedido cobrado”: se debe buscar partidas compensadas para factura asociada (documento de facturación -> BKPF/BSEG -> pago/clear) o usar tabla de compensación REGUH/BSAD. Este mapeo necesita definición.

- Vendedor
  - KNA1.VKBUR no existe: vendedor normalmente está en VBAK-VKGRP (grupo vendedor) o VBAP-POSNR? Vendedor por cliente es KNVP-KOAR o VBAK-BSTNK? En muchos sistemas el vendedor (sales employee) se guarda en VBAK-BSTNK? Más común es VBPA partner function 'AG' o 'VE' (sales person) o KNVP. Mapping propuesto “KNA1 VKBUR” es incorrecto.

4. Revisión de arquitectura técnica (qué construir)

Recomendación general: No construir como ALV clásico puro. Propongo construir una solución basada en CDS + Fiori (o RAP + Fiori Elements) y exponer transacción GUI mínima si el usuario lo demanda. Justificación:
- S/4HANA: CDS y RAP aprovechan el stack HANA (push-down), mejor rendimiento que SELECTs ABAP multi-join.
- Clean Core: usar CDS/RAP evita modificaciones directas a tablas y facilita extensibilidad (extensibility points) y uso de APIs publicadas.
- UX/Fiori: los usuarios principales usan Fiori; Fiori Elements + List Report + Object Page cubren "resumen/detalle", variantes, filtros, acciones y edición con floorplans estándar.
- Mantenibilidad y performance: CDS con associations y agregaciones (window functions) permite devolver solo lo necesario y calcular “última entrega/factura” eficientemente.

Opciones concretas (prioridad):
1) CDS-Beneficio + OData consumption para Fiori Elements (List report + Object page). Backend: CDS consumption view(s) que hagan los joins correctos y agregaciones por pedido; exponer en OData y crear una Fiori Elements app. Edición de fecha de entrega mediante un transaction-enabled RAP or BOPF/Business Object with update CDS/Service layer that calls standard BAPIs (e.g., BAPI_OUTB_DELIVERY_CHANGE) — nunca update directo a LIKP/LIPS por SQL.
2) Si cliente insiste en GUI ALV, construir un report técnico mínimo que consuma CDS view(s) como datasource (SELECT desde CDS) y entregue ALV; pero esto debe ser tratado como "fallback", con Fiori como target a mediano plazo.
3) Evitar leer KONV directamente; crear CDS que exponga pricing o consumir API SAP SD pricing.

No recomiendo:
- Acceso directo e indiscriminado a tablas cluster (KONV) ni actualización directa de tablas estándar.
- ALV clásico como solución principal en S/4HANA con usuarios Fiori (mala UX y peor escalabilidad).

5. Lista de aclaraciones necesarias (preguntas priorizadas)

Alcance / Estados / Reglas
1. Definir “estado general del pedido”: priorización entre VBUK, VBUP y estado de factura. Dar reglas explícitas (p.ej. si existe factura completa -> CLOSED; else if parcial delivery -> PARTIAL_DELIVERY; else OPEN).
2. Definir qué condiciones se consideran “pedido rechazado” y qué campo/valor identifica “anulado” (VBAK-LOEKZ? VBUK-ABRUF?).
3. ¿Qué significa “pedido cobrado”? ¿Cobrado = factura compensada (cleared) en FI? Proveer reglas concretas (buscar documentos de clearing por BELNR o buscar partidas en BSAD/BSIK).
4. Definir condición de precio a mostrar (PR00 / importe neto / subtotal / total con impuestos). Indicar moneda y escala de precios.

Datos / Mapeos
5. Confirmar qué campo identifica vendedor (VKGRP, KBUND? partner function VE in VBPA? KNVP?). Proporcionar ejemplo de pedido con vendedor.
6. ¿Se requiere detalle por posición en la vista principal o por drill-down? ¿Modo resumen debe agrupar por pedido y mostrar los últimos delivery/invoice?

UI / Edición
7. Confirmar requerimiento de editar fecha de entrega: ¿editar fecha de entrega en entrega existente (LIKP-WADAT_IST) o “prometida al cliente” en VBAP-EDATU? ¿Qué BAPI validar para cambios? Definir reglas de validación (no permitir fechas pasadas, bloqueo por autorizaciones).
8. Autorizaciones: proporcionar objeto de autorización requerido o reglas para crear PFCG. ¿Quién puede editar y con qué roles?

Performance / Volumen
9. Confirmación del volumen real y consultas típicas (número de usuarios concurrentes, filtros usuales). ¿Se espera exportar 100k filas a Excel frecuentemente?
10. ¿Perfil de hardware/DB? (S/4HANA cloud/OnPremise, versión) — para dimensionar CDS/indices.

Integraciones FI
11. ¿Se requiere integrar FI en la primera versión? Actualmente dice “no obligatorio”, pero hay campos (fecha cobro, doc contable). Confirmar prioridad.

Otros
12. ¿Requieren que el reporte se ejecute en background y entregue archivo? Formato y periodicidad.
13. Datos de prueba: solicitar ejemplos (pedidos con múltiples entregas/facturas, anulación, parcialmente entregados) para pruebas.

6. Propuesta técnica (concrete)

Arquitectura recomendada (top-level)
- Backend: crear un set de CDS views (core) en S/4 que expongan la información por pedido/posición:
  - ZC_SalesOrderHeader (consumption view) basado en I_SalesOrderHeader CDS o directamente en VBAK, exponiendo campos header.
  - ZC_SalesOrderItem (consumption) basado en VBAP con associations a MAKT (texto), KNA1/KNVV (cliente), VBUP (item status).
  - ZC_DeliveryLatest por pedido/posición: view que devuelve la última entrega (LIKP/LIPS) usando window function ROW_NUMBER() partition by sales order/item order by delivery date desc.
  - ZC_BillingLatest por pedido/posición: view que devuelve la última factura (VBRK/VBRP) similar.
  - ZC_Pricing: EXPOSER pricing principal usando SAP exposed pricing CDS or view / use API pricing; no leer KONV directamente.
  - ZC_PaymentStatus: view que relacione factura -> BKPF/BSEG/BSAD para identificar clearing and payment status.
- Aggregation: un consumption view que una header + item + latest delivery + latest billing + payment status con LEFT OUTER JOINs y asociaciones, evitando múltiples filas indeseadas. Para “una sola línea por pedido” crear una view agregada por VBAK (modo resumen) y otra para items (modo detalle).
- Exponer OData via CDS Consumption o RAP: Preferir RAP if need transactional editing (para edición de fecha de entrega). RAP te da framework para CUD con checks y BAdIs. Si se usa CDS + OData, las modificaciones deben llamar a BAPI/BUSIINESS API.

Edición de fecha de entrega
- No actualizar LIKP/LIPS directamente. Implementar acción que invoque BAPI (p.ej. BAPI_GOODSMVT_CHANGE no es adecuado; para change delivery use SD standard API like RV_DELIVERY_SOME? Confirm exact BAPI: BAPI_OUTB_DELIVERY_CHANGE (o BAPI_DELIVERYPROCESSING?) ), o usar BAdI/standard methods. Validaciones:
  - Chequear bloqueo/estado de entrega
  - Chequear autorización (PFCG)
  - Actualizar mediante API y refrescar CDS.

Estrategia de estados y semáforos
- Implementar lógica en CDS o en capa servicio: definir enumeración de estados (OPEN, PARTIAL_DELIVERY, COMPLETED, BLOCKED, REJECTED, CANCELLED) con reglas claras (prioridad definida). Los colores se puede manejar en Fiori UI (semantic colors) basados en el status expuesto por CDS.

Documentos múltiples
- Regla “mostrar la más reciente” implementada con window functions y seleccionando ROW_NUMBER = 1 por partition sales order/item.

Performance
- Push-down al HANA con CDS; evitar SELECTs loops. Crear índices/fields de selección (por ejemplo, campos usados en filtros: VBELN, KUNNR, MATNR, ERDAT, VKORG).
- Implementar paginación y límites por UI; evitar retorno de 100k filas de forma predeterminada. Para export masivo, ofrecer job background que cree archivo y almacenarlo.
- Usar agregaciones en BD para reporte resumen.
- Evitar lectura de KONV; usar CDS exposed pricing views o API.

Objetos ABAP a crear
- CDS entities (core + consumption) como arriba.
- Service binding OData (V2 o V4) o RAP business object (si se requiere edición).
- Fiori Elements List Report + Object Page app (UI5) (o un shell wrapper que permita abrir desde Launchpad).
- Program fallback Z_REPORT that calls CDS and shows ALV para usuarios legacy (opcional y con texto claro que será deprecated).
- Clase ABAP para llamadas a BAPIs de edición de delivery con manejo de errores/lock.

Navegación / drill-down
- List report -> object page por pedido. Desde línea de factura doble-clic: abrir app estándar de factura (Transaction display) o open via intent navigation a Fiori app de display document using semantic object. En GUI fallback use DOUBLE-CLICK event to call standard transactions (VF03, VL03N, VA03) with parameters.

Seguridad / Autorizaciones
- Usar checks AUTHORITY-CHECK con objeto de autorización estándar SD (V_VBAK, or V_VBPA) o definir objeto Z si necesario, y PFCG roles para edición. Documentar roles.

7. Estimación condicionada

¿Se puede estimar ya?
- No. La EF NO ALCANZA para estimar con fiabilidad.

Partes no estimables / dependencias que faltan
- Definición completa de estados y reglas de prioridad (bloqueo para desarrollo).
- Decisión sobre si la edición de la fecha de entrega será permitida en primera versión y qué API/BAPI usar (bloqueo).
- Definición de “pedido cobrado” y mapeo FI exacto (BSID/BSAD/BSEG/BKPF) — necesario para mapeo y performance.
- Aprobación de arquitectura (CDS+Fiori/RAP vs ALV clásico). Si cliente obliga ALV clásico puro, estimación cambia y se deben definir límite de datos y expectativas de respuesta.
- Entorno S/4HANA versión (Cloud/OnPremise) para elegir RAP vs OData V2.

Estimación tentativa (solo en caso de aprobar CDS+Fiori & clarificaciones resueltas)
- Diseño y CDS views core + consumption: 5-8 días
- Implementación de lógica “última entrega/factura” y pruebas unitarias: 3-5 días
- Servicio OData / RAP + binding: 4-6 días
- Fiori Elements app (list + object page) + authorization + navigation: 6-8 días
- Integración de edición de fecha de entrega via BAPI y pruebas: 4-6 días
- Pruebas integrales, ajustes y documentación: 4-6 días
Total aproximado: 26-39 días hábiles (condicionado a clarificaciones y disponibilidad de datos de prueba). Si se exige ALV clásico puro para versión 1, estimación baja pero con menor calidad y mayor riesgo de re-work.

Conclusión breve y recomendaciones finales
- No recomiendo construir esta solución únicamente como ALV clásico. Es técnicamente posible pero con alto riesgo de problemas de performance, mantenimiento y no alineación con S/4HANA / Fiori.
- Propongo rediseñar como CDS consumption views + Fiori Elements (o RAP si se requiere CUD) y exponer un fallback ALV solo como opción temporal.
- La EF no está lista para desarrollo: faltan definiciones concretas de estados, mapeos FI, definición de vendedor, reglas de edición y autorización. Es imprescindible responder las preguntas de la sección 5 antes de estimar y comenzar el desarrollo.

Si quieres, preparo:
- Un diagrama de datos con joins correctos y ejemplos de CDS (esqueleto) para cada view propuesta.
- Un borrador de reglas de estado (tabla con priorización) que puedas validar con funcional.